# 巴西电商用户行为分析

#### 本项目使用巴西电商公开数据集，分析用户行为与订单特征。主要步骤：
1. 数据清洗与预处理（处理缺失值、统一时间格式）
2. 特征工程（构建用户、订单、产品维度特征）
3. 可视化分析与业务洞察

# 第一部分：巴西电商用户行为分析 - 数据清洗
## 1.数据预处理
### 1.1加载文件和统一时间格式
### 1.2了解各表基本情况,检查是否有缺失值
### 1.3检查有缺失值的表
### 1.4处理缺失值
#### 1.4.1 处理orders订单表的缺失值
#### 1.4.2 处理products表的缺失值
### 1.5 检查数值字段的数据类型
确保金额、数量等字段为数值类型，以便后续计算。
### 1.6 检查金额字段的异常值
确保金额不为负数
### 1.7 将清洗后的表保存为新文件，供第二部分（特征工程）使用
### 1.8 数据清洗总结

#####

## 1. 数据预处理
（统一时间格式、探查数据基础信息、处理缺失值）

In [97]:
import pandas as pd
import numpy as np

#####

## 1.1加载文件和统一时间格式

In [51]:
customers = pd.read_csv('brazil_ecommerce_project/olist_customers_dataset.csv')  #客户数据

In [52]:
geolocation = pd.read_csv('brazil_ecommerce_project/olist_geolocation_dataset.csv') #地理位置数据

In [53]:
order_items = pd.read_csv('brazil_ecommerce_project/olist_order_items_dataset.csv'
                          ,parse_dates=['shipping_limit_date']) #订单项目数据

In [54]:
order_payments = pd.read_csv('brazil_ecommerce_project/olist_order_payments_dataset.csv') #订单付款数据

In [55]:
order_reviews = pd.read_csv('brazil_ecommerce_project/olist_order_reviews_dataset.csv'
                            ,parse_dates=['review_creation_date','review_answer_timestamp']) 
#订单评分数据

In [56]:
orders = pd.read_csv('brazil_ecommerce_project/olist_orders_dataset.csv'
                    ,parse_dates=['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date', 
             'order_estimated_delivery_date']) #订单数据

In [57]:
products = pd.read_csv('brazil_ecommerce_project/olist_products_dataset.csv') #产品数据

In [58]:
sellers = pd.read_csv('brazil_ecommerce_project/olist_sellers_dataset.csv')  #卖家数据

In [59]:
product_category= pd.read_csv('brazil_ecommerce_project/product_category_name_translation.csv') 
#商品类别名称翻译映射表

#####

## 1.2了解各表基本情况,检查是否有缺失值

In [60]:
#客户表的基本情况
customers.info()
customers.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


,customer_zip_code_prefix
count,99441.000000
mean,35137.474583
std,29797.938996
min,1003.000000
25%,11347.000000
50%,24416.000000
75%,58900.000000
max,99990.000000


### 客户表探查结论：共 99441 条记录，无缺失值，字段类型正常，可用于后续分析。

In [61]:
#了解地理位置表的基本情况
geolocation.info()
geolocation.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06,1.000163e+06
mean,3.657417e+04,-2.117615e+01,-4.639054e+01
std,3.054934e+04,5.715866e+00,4.269748e+00
min,1.001000e+03,-3.660537e+01,-1.014668e+02
25%,1.107500e+04,-2.360355e+01,-4.857317e+01
50%,2.653000e+04,-2.291938e+01,-4.663788e+01
75%,6.350400e+04,-1.997962e+01,-4.376771e+01
max,9.999000e+04,4.506593e+01,1.211054e+02


### 地理位置表探查结论：共 1000163 条记录，无缺失值，字段类型正常，可用于后续分析。

In [62]:
#了解订单项目表的基本情况
order_items.info()
order_items.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  object        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  object        
 3   seller_id            112650 non-null  object        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 6.0+ MB


,order_item_id,shipping_limit_date,price,freight_value
count,112650.000000,112650,112650.000000,112650.000000
mean,1.197834,2018-01-07 15:36:52.192685312,120.653739,19.990320
min,1.000000,2016-09-19 00:15:34,0.850000,0.000000
25%,1.000000,2017-09-20 20:57:27.500000,39.900000,13.080000
50%,1.000000,2018-01-26 13:59:35,74.990000,16.260000
75%,1.000000,2018-05-10 14:34:00.750000128,134.900000,21.150000
max,21.000000,2020-04-09 22:35:08,6735.000000,409.680000
std,0.705124,NaN,183.633928,15.806405


### 订单项目表探查结论：共 112650 条记录，无缺失值，字段类型正常，可用于后续分析。

In [63]:
#了解订单付款表的基本情况
order_payments.info()
order_payments.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


### 订单付款表探查结论：共 103886 条记录，无缺失值，字段类型正常，可用于后续分析。

In [64]:
#了解订单评分表的基本情况
order_reviews.info()
order_reviews.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  object        
 1   order_id                 99224 non-null  object        
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  object        
 4   review_comment_message   40977 non-null  object        
 5   review_creation_date     99224 non-null  datetime64[ns]
 6   review_answer_timestamp  99224 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(4)
memory usage: 5.3+ MB


,review_score,review_creation_date,review_answer_timestamp
count,99224.000000,99224,99224
mean,4.086421,2018-01-12 20:49:23.948238336,2018-01-16 00:23:56.977938688
min,1.000000,2016-10-02 00:00:00,2016-10-07 18:32:28
25%,4.000000,2017-09-23 00:00:00,2017-09-27 01:53:27.249999872
50%,5.000000,2018-02-02 00:00:00,2018-02-04 22:41:47.500000
75%,5.000000,2018-05-16 00:00:00,2018-05-20 12:11:21.500000
max,5.000000,2018-08-31 00:00:00,2018-10-29 12:27:35
std,1.347579,NaN,NaN


### 订单评分表探查结论：共 99224  条记录，有缺失值，等待后续处理。

In [65]:
#了解订单表的基本情况
orders.info()
orders.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99281,97658,96476,99441
mean,2017-12-31 08:43:12.776581120,2017-12-31 18:35:24.098800128,2018-01-04 21:49:48.138278656,2018-01-14 12:09:19.035542272,2018-01-24 03:08:37.730111232
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00
25%,2017-09-12 14:46:19,2017-09-12 23:24:16,2017-09-15 22:28:50.249999872,2017-09-25 22:07:22.249999872,2017-10-03 00:00:00
50%,2018-01-18 23:04:36,2018-01-19 11:36:13,2018-01-24 16:10:58,2018-02-02 19:28:10.500000,2018-02-15 00:00:00
75%,2018-05-04 15:42:16,2018-05-04 20:35:10,2018-05-08 13:37:45,2018-05-15 22:48:52.249999872,2018-05-25 00:00:00
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12 00:00:00


### 订单表探查结论：共 99441 条记录，有缺失值，等待后续处理。

In [66]:
#了解产品表的基本情况
products.info()
products.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


### 产品表探查结论：共 32951 条记录，有缺失值，等待后续处理。

In [67]:
#了解卖家表的基本情况
sellers.info()
sellers.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


,seller_zip_code_prefix
count,3095.000000
mean,32291.059451
std,32713.453830
min,1001.000000
25%,7093.500000
50%,14940.000000
75%,64552.500000
max,99730.000000


### 商家表探查结论：共 3095 条记录，无缺失值，字段类型正常，可用于后续分析。

In [68]:
#了解产品类别表的基本情况
product_category.info()
product_category.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1


### 产品类别探查结论：共 71 条记录，无缺失值，字段类型正常，可用于后续分析。

#####

## 1.3检查有缺失值的表

In [69]:
order_reviews.isna().sum()
#订单评分表中有缺失值，查找原因后发现，用户可能选择只打分而不写文字评论，所以缺失是正常现象，无需处理此表的缺失值。

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [70]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [71]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

#####

## 1.4处理缺失值

### 1.4.1 处理orders订单表的缺失值

In [72]:
# 1.分析缺失值与订单状态的关系（按订单状态分组，计算缺失值列的缺失数量）：
missing_by_status = orders.groupby('order_status')[['order_approved_at', 
'order_delivered_carrier_date','order_delivered_customer_date']].apply(lambda x : x.isna().sum())

In [73]:
missing_by_status
#经发现：
# canceled / created / unavailable：所有时间戳缺失都是合理的，无需处理。
# invoiced / processing / approved：只有批准时间存在，后续缺失合理。
# shipped：已发货但未送达，因此 customer_date 缺失合理。
# 状态为 delivered 的订单仍有少量缺失：order_approved_at 缺 14 条，order_delivered_carrier_date 缺 2 条，
# order_delivered_customer_date 缺 8 条。这些订单状态标记为“已送达”，但关键时间戳缺失，属于数据质量问题。
# 不填充，因为这些缺失无法合理推测。

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [74]:
# 2.用布尔特征标记流程是否完成，方便后续筛选分析：
# 比如分析不同阶段的转化率（如批准率、发货率），计算配送时间时，只筛选 has_delivered 为 True 的行。
orders['has_approved'] = orders['order_approved_at'].notna()
orders['has_shipped'] = orders['order_delivered_carrier_date'].notna()
orders['has_delivered'] = orders['order_delivered_customer_date'].notna()

In [75]:
# 3.标记 delivered 状态下，送达日期缺失的8个异常订单：
# 标记 delivery 状态下的异常订单,这里只用 order_delivered_customer_date 缺失作为异常标志，因为送达日期最关键。
orders['yichang_delivery'] = (
    (orders['order_status'] == 'delivered') & 
    (orders['order_delivered_customer_date'].isna())
)

In [76]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,has_approved,has_shipped,has_delivered,yichang_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,True,True,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,True,True,True,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,True,True,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,True,True,True,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,True,True,True,False


### 1.4.2 处理products表的缺失值

In [77]:
# 1.分析缺失模式
missing_cat = products[products['product_category_name'].isna()]
len(missing_cat)

610

In [78]:
missing_cat
# 发现缺失类别的产品在其他字段，如重量、长宽高字段也有值，说明它们是真实存在的产品，只是没有分类信息。
# product_name_length、product_description_length、product_photos_qty 
# 这三个字段的缺失是和类别缺失完全一致的（都是610条）。

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [79]:
# 2.处理产品类别缺失值:
# 创建'unknown'类别
products['product_category_name'] = products['product_category_name'].fillna('unknown')
# 创建一个布尔特征标记那些原本不缺失的,在分析时，可以选择是否包含这些“unknown”类别
products['has_product_category_name'] = products['product_category_name'] != 'unknown'

In [80]:
# 3.处理类别缺失导致的连锁缺失：
# 如果要分析描述长度与销量的关系，可以用0填充缺失，代表“无描述”
products['product_description_lenght'] = products['product_description_lenght'].fillna(0)
# 用布尔特征标记原本描述长度字段缺失的值
products['has_no_description'] = products['product_description_lenght'] == 0

In [81]:
# 4.处理重量和尺寸的少量缺失（各2条）
# 这2条缺失很可能是录入遗漏，由于数量极少，可以用同类产品的均值填充：
# 同一类别的产品，往往在物理属性（重量、尺寸）上有一定相似性。例如，所有“手机”的重量通常在 100-300 克之间，
# 而“冰箱”则可能几十公斤。用同类均值比用全局均值（所有产品的平均）更贴近真实情况。
# 产品表中的重量、尺寸等字段分别只有 2 个缺失值，用同类均值填充既符合业务直觉，又不会引入太大偏差，所以是很好的选择。
avg_weight_length_height_width = products.groupby(
    'product_category_name')[['product_weight_g',
                              'product_length_cm',
                              'product_height_cm',
                              'product_width_cm']].transform('mean')

cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in cols:
    products[col] = products[col].fillna(avg_weight_length_height_width[col])

### 验证填充结果
检查重量、尺寸字段是否还有缺失值，确保所有缺失已被处理。

In [82]:
# 验证填充结果，缺失值已经全部填充
products[cols].isna().sum()

product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64

#####

## 1.5 检查数值字段的数据类型
确保金额、数量等字段为数值类型，以便后续计算。

In [83]:
customers.dtypes

customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

### customers表的结论：customer_zip_code_prefix是整数，没问题。

In [84]:
geolocation.dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

### geolocation表的结论：编码字段是整数没问题，经纬度是浮点数也没问题。

In [85]:
order_items.dtypes

order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

### order_items表的结论：order_item_id 是整数，没问题；price和freight_values是浮点数，没问题。

In [86]:
order_payments.dtypes

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

### order_payments表的结论：payment_sequential和payment_installments 是整数，没问题；payment_value 是浮点数，没问题。

In [87]:
order_reviews.dtypes

review_id                          object
order_id                           object
review_score                        int64
review_comment_title               object
review_comment_message             object
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

### order_reviews表的结论：review_score 是整数，没问题。

In [88]:
orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
has_approved                               bool
has_shipped                                bool
has_delivered                              bool
yichang_delivery                           bool
dtype: object

### orders表的结论：无数值类字段。

In [89]:
products.dtypes

product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
has_product_category_name        bool
has_no_description               bool
dtype: object

### products表的结论：产品名字长度、数量、重量、尺寸字段均为浮点数，没问题。

In [90]:
sellers.dtypes

seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object

### sellers表的结论：seller_zip_code_prefix为整数，没问题。

In [91]:
product_category.dtypes

product_category_name            object
product_category_name_english    object
dtype: object

### product_category表的结论：无数值类字段。

#####

## 1.6 检查金额字段的异常值
确保金额不为负数

In [92]:
# 经检查，发现order_items表和order_payments表中有金额字段。
# 检查订单项表中是否有负的金额
order_items[order_items['price'] < 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [93]:
order_items[order_items['freight_value'] < 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [94]:
# 检查支付表中是否有负的支付金额
order_payments[order_payments['payment_value'] < 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


### 结论：所有金额字段的数值正常。

#####

## 1.7 将清洗后的表保存为新文件，供第二部分（特征工程）使用

In [95]:
# 保存清洗后的订单表
orders.to_csv('brazil_ecommerce_project/cleaned_orders.csv',  index=False)

In [96]:
# 保存清洗后的产品表
products.to_csv('brazil_ecommerce_project/cleaned_products.csv', index=False)

#####

## 1.8 数据清洗总结
- 共处理了 3 个有缺失值的表：订单表、评分表、产品表。
- 订单表：增加了 3 个布尔特征标记流程，并标记了 8 条异常订单。
- 评分表：保留文本缺失，未做处理。
- 产品表：填充了类别缺失（610 条），用 0 填充了描述长度，用同类均值填充了重量和尺寸（各 2 条）。
- 所有表已清洗完毕，可进入特征工程阶段。